In [1]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")

TRAIN_DIR = PROJECT_ROOT / "data" / "alphabet" / "raw" / "Train"
TEST_DIR = PROJECT_ROOT / "data" / "alphabet" / "raw" / "Test"

LANDMARKS_DIR = PROJECT_ROOT / "data" / "alphabet" / "landmarks"

TRAIN_LANDMARKS_CSV = LANDMARKS_DIR / "train_landmarks_normalized.csv"
TEST_LANDMARKS_CSV = LANDMARKS_DIR / "test_landmarks_normalized.csv"

TRAIN_FAILED_CSV = LANDMARKS_DIR / "train_failed.csv"
TEST_FAILED_CSV = LANDMARKS_DIR / "test_failed.csv"

TRAIN_STATS_CSV = LANDMARKS_DIR / "train_stats_rebuilt.csv"
TEST_STATS_CSV = LANDMARKS_DIR / "test_stats_rebuilt.csv"

print("Train dir exists:", TRAIN_DIR.exists())
print("Test dir exists:", TEST_DIR.exists())
print("Train landmarks exists:", TRAIN_LANDMARKS_CSV.exists())
print("Test landmarks exists:", TEST_LANDMARKS_CSV.exists())

Train dir exists: True
Test dir exists: True
Train landmarks exists: True
Test landmarks exists: True


In [2]:
train_existing = pd.read_csv(TRAIN_LANDMARKS_CSV)
test_existing = pd.read_csv(TEST_LANDMARKS_CSV)

print("Train shape:", train_existing.shape)
print("Test shape:", test_existing.shape)

print("Train columns:")
print(train_existing.columns.tolist())

display(train_existing.head())

Train shape: (4827, 66)
Test shape: (695, 66)
Train columns:
['x0', 'y0', 'z0', 'x1', 'y1', 'z1', 'x2', 'y2', 'z2', 'x3', 'y3', 'z3', 'x4', 'y4', 'z4', 'x5', 'y5', 'z5', 'x6', 'y6', 'z6', 'x7', 'y7', 'z7', 'x8', 'y8', 'z8', 'x9', 'y9', 'z9', 'x10', 'y10', 'z10', 'x11', 'y11', 'z11', 'x12', 'y12', 'z12', 'x13', 'y13', 'z13', 'x14', 'y14', 'z14', 'x15', 'y15', 'z15', 'x16', 'y16', 'z16', 'x17', 'y17', 'z17', 'x18', 'y18', 'z18', 'x19', 'y19', 'z19', 'x20', 'y20', 'z20', 'label', 'file_path', 'split']


,x0,y0,z0,x1,y1,z1,x2,y2,z2,x3,...,z18,x19,y19,z19,x20,y20,z20,label,file_path,split
0,0.0,0.0,0.0,-0.285079,-0.005470,-0.132447,-0.536141,-0.280805,-0.154404,-0.610316,...,-0.198524,0.162542,-0.440788,-0.159892,0.153593,-0.346873,-0.091791,A,../data/alphabet/raw/train\A\Image_1685009115....,train
1,0.0,0.0,0.0,0.237101,-0.059265,-0.128390,0.451640,-0.331519,-0.153078,0.518450,...,-0.073203,-0.167767,-0.454129,-0.026119,-0.151433,-0.394190,0.035014,A,../data/alphabet/raw/train\A\Image_1685009117....,train
2,0.0,0.0,0.0,-0.067200,0.149242,-0.089536,-0.051379,0.373939,-0.221029,0.040574,...,-0.405370,0.247083,0.456774,-0.364406,0.249568,0.501492,-0.339672,A,../data/alphabet/raw/train\A\Image_1685009120....,train
3,0.0,0.0,0.0,0.252646,-0.143031,-0.062913,0.443554,-0.428070,-0.048246,0.485227,...,0.017269,-0.185673,-0.558414,0.028644,-0.144139,-0.443946,0.066502,A,../data/alphabet/raw/train\A\Image_1685009122....,train
4,0.0,0.0,0.0,-0.265195,-0.155066,-0.083215,-0.369033,-0.462475,-0.076585,-0.340364,...,0.032981,0.179019,-0.755773,0.022316,0.199794,-0.608524,0.053437,A,../data/alphabet/raw/train\A\Image_1685009125....,train


Rebuilding failed files


In [3]:
def rebuild_failed_files_by_filename(input_dir, landmarks_csv_path, split_name):
    existing_df = pd.read_csv(landmarks_csv_path)

    if "label" not in existing_df.columns:
        raise ValueError("The landmarks CSV must contain a 'label' column.")

    if "file_path" not in existing_df.columns:
        raise ValueError("The landmarks CSV must contain a 'file_path' column.")

    successful_keys = set()

    for _, row in existing_df.iterrows():
        saved_label = str(row["label"])
        saved_file_name = os.path.basename(str(row["file_path"]))

        successful_keys.add((saved_label, saved_file_name))

    failed_files = []
    stats = {}

    for label in sorted(os.listdir(input_dir)):
        label_path = input_dir / label

        if not label_path.is_dir():
            continue

        total = 0
        kept = 0
        failed = 0

        for file_name in sorted(os.listdir(label_path)):
            file_path = label_path / file_name

            if not file_path.is_file():
                continue

            total += 1

            key = (label, file_name)

            if key in successful_keys:
                kept += 1
            else:
                failed += 1

                failed_files.append({
                    "label": label,
                    "file_path": str(file_path),
                    "split": split_name
                })

        stats[label] = {
            "total": total,
            "kept": kept,
            "failed": failed,
            "keep_rate": kept / total if total > 0 else 0
        }

    failed_df = pd.DataFrame(failed_files)
    stats_df = pd.DataFrame(stats).T.reset_index().rename(columns={"index": "label"})

    return failed_df, stats_df

Generate train/test failed CSVs

In [4]:
train_failed, train_stats = rebuild_failed_files_by_filename(
    TRAIN_DIR,
    TRAIN_LANDMARKS_CSV,
    "train"
)

test_failed, test_stats = rebuild_failed_files_by_filename(
    TEST_DIR,
    TEST_LANDMARKS_CSV,
    "test"
)

print("Train failed:", len(train_failed))
print("Test failed:", len(test_failed))

display(train_stats.sort_values("keep_rate"))
display(test_stats.sort_values("keep_rate"))

Train failed: 5663
Test failed: 1105


,label,total,kept,failed,keep_rate
21,W,439.0,74.0,365.0,0.168565
16,R,438.0,79.0,359.0,0.180365
20,V,430.0,88.0,342.0,0.204651
11,M,433.0,112.0,321.0,0.258661
22,X,443.0,127.0,316.0,0.286682
1,B,443.0,131.0,312.0,0.295711
19,U,436.0,139.0,297.0,0.318807
2,C,437.0,140.0,297.0,0.320366
13,O,440.0,141.0,299.0,0.320455
17,S,443.0,170.0,273.0,0.383747


,label,total,kept,failed,keep_rate
21,W,75.0,1.0,74.0,0.013333
19,U,75.0,1.0,74.0,0.013333
22,X,75.0,3.0,72.0,0.040000
16,R,75.0,6.0,69.0,0.080000
20,V,75.0,8.0,67.0,0.106667
3,D,75.0,9.0,66.0,0.120000
13,O,75.0,13.0,62.0,0.173333
11,M,75.0,14.0,61.0,0.186667
2,C,75.0,19.0,56.0,0.253333
1,B,75.0,20.0,55.0,0.266667


In [5]:
train_failed.to_csv(TRAIN_FAILED_CSV, index=False)
test_failed.to_csv(TEST_FAILED_CSV, index=False)

train_stats.to_csv(TRAIN_STATS_CSV, index=False)
test_stats.to_csv(TEST_STATS_CSV, index=False)

print("Saved:")
print(TRAIN_FAILED_CSV)
print(TEST_FAILED_CSV)
print(TRAIN_STATS_CSV)
print(TEST_STATS_CSV)

Saved:
..\data\alphabet\landmarks\train_failed.csv
..\data\alphabet\landmarks\test_failed.csv
..\data\alphabet\landmarks\train_stats_rebuilt.csv
..\data\alphabet\landmarks\test_stats_rebuilt.csv
